In [ ]:
# Import all necessary packages
import numpy as np
import pandas as pd
from statannotations.Annotator import Annotator
import matplotlib.pyplot as plt
import seaborn as sns
import distinctipy
import matplotlib.colors as mcolors
from statsmodels.stats.multitest import multipletests
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import mannwhitneyu
import ast
import rootutils

# Set directories and load data
path_root = str(rootutils.find_root(indicator=".project-root"))
path_plots = f"{path_root}/plots"
path_data = f"{path_root}/data/age-regression"
df_feats = pd.read_excel(f"{path_data}/features.xlsx", index_col=0)
df = pd.read_excel(f"{path_data}/data.xlsx")

statuses_rename = {x: x.replace(' ', '\n').replace('-', '\n') for x in df['Status'].value_counts().index.values}
df['Status Origin'] = df['Status']
df['Status'] = df['Status'].replace(statuses_rename)
status_count = df['Status'].value_counts()
statuses = status_count.index.values

df_groups = pd.read_excel(f"{path_data}/groups.xlsx", index_col=0)
icd_chpts = np.sort(df_groups['ICD-11 chapter'].unique())
icd_cols = []
for icd_chpt in icd_chpts:
    icd_cols.append(f'Passed\nICD-11\nChapter {icd_chpt}')
icd_cols_max = [f"Max\n{x}" for x in icd_cols]

# Prepare colors
def make_rgb_transparent(rgb, bg_rgb, alpha):
    return [alpha * c1 + (1 - alpha) * c2 for (c1, c2) in zip(rgb, bg_rgb)]

# Colors for ICD-11 chapters
colors = distinctipy.get_colors(len(icd_chpts), [mcolors.hex2color(mcolors.CSS4_COLORS['black']), mcolors.hex2color(mcolors.CSS4_COLORS['white'])], rng=1337, pastel_factor=0.5)
colors_icd_chpts = {icd_chpt: colors[icd_chpt_id] for icd_chpt_id, icd_chpt in enumerate(icd_chpts)}
colormaps_icd_chpts = {
    icd_chpt: LinearSegmentedColormap.from_list(
        name=f"ICD-11 Chapter {icd_chpt} cmap",
        colors=[make_rgb_transparent(colors_icd_chpts[icd_chpt], (1, 1, 1), 0.2), colors_icd_chpts[icd_chpt]], N=256
    )
    for icd_chpt in icd_chpts
}

# Colors for statuses
colors = distinctipy.get_colors(len(statuses) - 1, [mcolors.hex2color(mcolors.CSS4_COLORS['dodgerblue']), mcolors.hex2color(mcolors.CSS4_COLORS['white']), mcolors.hex2color(mcolors.CSS4_COLORS['black'])], rng=1337)
colors_status = {'Control': mcolors.hex2color(mcolors.CSS4_COLORS['dodgerblue'])}
for status_id, status in enumerate(statuses):
    if status != 'Control':
        colors_status[status] = colors[status_id - 1]

# Create dataframe for the figure
df_clocks = pd.read_excel(f"{path_data}/clocks_meta.xlsx", index_col=0, nrows=1)
new_cols = ['Passed\nICD-11\nTotal'] + icd_cols + ['Max\nPassed\nICD-11\nTotal'] + icd_cols_max
for col in new_cols: 
    df_clocks[col] = None
    
df['EpInflammAge Error'] = df['EpInflammAge'] - df['Age']
        
for section_id, section_row in df_groups.iterrows():
    section_statuses = ast.literal_eval(section_row['Statuses'])
    section_groups = ast.literal_eval(section_row['Groups'])
    section_directions = ast.literal_eval(section_row['Directions'])
    df_section = df.loc[(df['GSE'] == section_row['GSE']) & (df['Status Origin'].isin(section_statuses)), ['Status Origin', 'EpInflammAge Error']]
        
    for section_group_id, section_group in enumerate(section_groups):
        _, pval = mannwhitneyu(
            df_section.loc[df_section["Status Origin"] == section_group[0], 'EpInflammAge Error'].values,
            df_section.loc[df_section["Status Origin"] == section_group[1], 'EpInflammAge Error'].values,
            alternative="two-sided",
        )
        bias_0 = np.mean(df_section.loc[df_section['Status Origin'] == section_group[0], 'EpInflammAge Error'])
        bias_1 = np.mean(df_section.loc[df_section['Status Origin'] == section_group[1], 'EpInflammAge Error'])
            
        df_clocks.at['EpInflammAge', f"pval\n{section_id}\n{section_group}"] = pval
        df_clocks.at['EpInflammAge', f"bias_0\n{section_id}\n{section_group}"] = bias_0
        df_clocks.at['EpInflammAge', f"bias_1\n{section_id}\n{section_group}"] = bias_1

# FDR correction
pvals_cols = [col for col in df_clocks.columns if 'pval' in col]
_, df_clocks.loc['EpInflammAge', pvals_cols], _, _ = multipletests(df_clocks.loc['EpInflammAge', pvals_cols], 0.05, method='fdr_bh')

passed_icd_chpt = {icd_chpt: 0 for icd_chpt in icd_chpts}
passed_icd_chpt_max = {icd_chpt: 0 for icd_chpt in icd_chpts}
for icd_chpt in icd_chpts:
    df_chpt = df_groups[df_groups['ICD-11 chapter'] == icd_chpt]
    for section_id, section_row in df_chpt.iterrows():
        section_statuses = ast.literal_eval(section_row['Statuses'])
        section_groups = ast.literal_eval(section_row['Groups'])
        section_directions = ast.literal_eval(section_row['Directions'])
            
        for section_group_id, section_group in enumerate(section_groups):
            passed_icd_chpt_max[icd_chpt] += 1
                
            pval = df_clocks.at['EpInflammAge', f"pval\n{section_id}\n{section_group}"]
            bias_0 = df_clocks.at['EpInflammAge', f"bias_0\n{section_id}\n{section_group}"]
            bias_1 = df_clocks.at['EpInflammAge', f"bias_1\n{section_id}\n{section_group}"]
                
            group_direction = section_directions[section_group_id]
            if pval < 0.05:
                if group_direction == 'Increasing' and bias_1 > bias_0:
                    passed_icd_chpt[icd_chpt] += 1
                elif group_direction == 'Decreasing' and bias_1 < bias_0:
                    passed_icd_chpt[icd_chpt] += 1
    df_clocks.at['EpInflammAge', f'Passed\nICD-11\nChapter {icd_chpt}'] = passed_icd_chpt[icd_chpt]
    df_clocks.at['EpInflammAge', f'Max\nPassed\nICD-11\nChapter {icd_chpt}'] = passed_icd_chpt_max[icd_chpt]              

df_clocks.at['EpInflammAge', f'Passed\nICD-11\nTotal'] = sum(passed_icd_chpt.values())
df_clocks.at['EpInflammAge', f'Max\nPassed\nICD-11\nTotal'] = sum(passed_icd_chpt_max.values())   

# Plot Supplementary Figure S3
mosaic_violins = []
mosaic_rows = np.sort(df_groups['Row on violin plot'].unique())
for mosaic_row in mosaic_rows:
    df_mosaic_row = df_groups[df_groups['Row on violin plot'] == mosaic_row].sort_values(by=['ICD-11 chapter', 'ICD-11 code', 'GSE'], ascending=[True, True, True])
    mosaic_row_labels = []
    for plot_id, plot_row in df_mosaic_row.iterrows():
        n_violins = len(ast.literal_eval(plot_row['Statuses']))
        mosaic_row_labels += [plot_id]*n_violins
    mosaic_violins.append(mosaic_row_labels)

max_mosaic_row = max([len(x) for x in mosaic_violins])
violons_empty_panels = set()
for row_id, row in enumerate(mosaic_violins):
    for added_spaces in range(len(row), max_mosaic_row):
        violons_empty_panels.add(f'Empty row {row_id}')
        mosaic_violins[row_id].append(f'Empty row {row_id}')

sns.set_theme(style='ticks')
fig_height = 5 * len(mosaic_violins)
fig_width = 1.4 * max_mosaic_row
fig, axs = plt.subplot_mosaic(mosaic=mosaic_violins, figsize=(fig_width, fig_height), gridspec_kw={}, sharey=False, sharex=False)

for plot_id, plot_row in df_groups.iterrows():

    plot_statuses = [statuses_rename[x] for x in ast.literal_eval(plot_row['Statuses'])]
    plot_groups_raw = ast.literal_eval(plot_row['Groups'])
    plot_groups = [(statuses_rename[x[0]], statuses_rename[x[1]]) for x in ast.literal_eval(plot_row['Groups'])]

    df_plot = df.loc[(df['GSE'] == plot_row['GSE']) & (df['Status'].isin(plot_statuses)), ['Status', 'EpInflammAge Error']]
    plot_status_count = df_plot['Status'].value_counts()
    plot_statuses_rename = {}
    for x in plot_statuses:
        plot_statuses_rename[x] = x + f"\nCount: {plot_status_count[x]}\nBias: {np.mean(df_plot.loc[df_plot['Status'] == x, 'EpInflammAge Error']):0.1f}"
    plot_statuses = [plot_statuses_rename[x] for x in plot_statuses]
    plot_groups = [(plot_statuses_rename[x[0]], plot_statuses_rename[x[1]]) for x in plot_groups]
    df_plot['Status'] = df_plot['Status'].replace(plot_statuses_rename)
    colors_plot_status = {plot_statuses_rename[x]: colors_status[x] for x in plot_statuses_rename}

    pval_formatted = []
    for plot_group_id, plot_group in enumerate(plot_groups):
        pval = df_clocks.at['EpInflammAge', f"pval\n{plot_id}\n{plot_groups_raw[plot_group_id]}"]
        pval_formatted.append(f"{pval:.1e}")

    violinplot = sns.violinplot(
        data=df_plot,
        x='Status',
        y='EpInflammAge Error',
        palette=colors_plot_status,
        scale='width',
        order=plot_statuses,
        saturation=0.75,
        legend=False,
        ax=axs[plot_id]
    )
    annotator = Annotator(
        ax=axs[plot_id],
        pairs=plot_groups,
        data=df_plot,
        x="Status",
        y="EpInflammAge Error",
        order=plot_statuses,
    )
    annotator.set_custom_annotations(pval_formatted)
    annotator.configure(loc='inside', verbose=0)
    annotator.annotate()

    axs[plot_id].set_xlabel('')
    axs[plot_id].set_ylabel('Age acceleration')
    if plot_row['GSE'] == 'GSEUNN':
        axs[plot_id].set_title(f"This work ({df_plot.shape[0]})")
    else:
        axs[plot_id].set_title(f"{plot_row['GSE']} ({df_plot.shape[0]})")
    axs[plot_id].set_facecolor(make_rgb_transparent(colors_icd_chpts[plot_row['ICD-11 chapter']], (1, 1, 1), 0.33))

for empty_panel in violons_empty_panels:
    axs[empty_panel].axis('off')
fig.tight_layout()
fig.savefig(f"{path_plots}/supplementary-figure-s3.png", bbox_inches='tight', dpi=200)
fig.savefig(f"{path_plots}/supplementary-figure-s3.pdf", bbox_inches='tight')
plt.close(fig)